In [26]:
from pathlib import Path
import pandas as pd
import json

In [27]:
site_to_camera = {
    "St.David": 7,
    "Contention": 6,
    "Fairbank": 1,
    "Boquillas": 8,
    "CharlestonMesquite": 2,
    "Moson": 3,
    "Hunter": 4,
    "Hereford": 5,
}
sites = list(site_to_camera.keys())

In [31]:
def longest_consecutive(mask): # maximum consecutive run of true
    if mask.dtype != bool:
        mask = mask.astype(bool)
    if mask.size == 0:
        return 0
    id = (mask != mask.shift()).cumsum() # group id
    max = 0
    start = None
    for g, sub in mask.groupby(id):
        if sub.iloc[0]:
            run = int(sub.sum())
            if run > max:
                max = run
                start = sub.index[0]
    return max, start

def missing_metrics(series, original):
    if series.index.duplicated().any():
        series = series[~series.index.duplicated(keep='first')]
    s = series.reindex(original)
    isna = s.isna()
    missing = float(isna.mean())
    longest, sd = longest_consecutive(isna)
    monthly = isna.groupby(isna.index.month).mean()
    max_month = float(monthly.max())
    med_month = float(monthly.median())
    if longest >= 365:
        gap_type = "systematic"
    elif (max_month > 0.6) and ((max_month - med_month) > 0.25):
        gap_type = "seasonal"
    else:
        gap_type = "random"
    return missing, longest, gap_type, monthly.to_dict(), sd

def missingness(feature, fc, full):
    if feature.index.duplicated().any():
        feature = feature[~feature.index.duplicated(keep='first')]
    if fc.index.duplicated().any():
        fc = fc[~fc.index.duplicated(keep='first')]
    f = feature.reindex(full) # reindex to match full series
    fc = fc.reindex(full)
    results = {}
    if fc.isna().all():
        return results
    for c in sorted(fc.dropna().unique()): # iterate over flow codes
        mask = (fc == c)
        if mask.sum() == 0:
            results[int(c)] = None
        else:
            results[int(c)] = float(f[mask].isna().sum() / mask.sum())
    return results

In [32]:
def main(dir: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    full = pd.date_range("2006-01-01", "2025-7-11", freq = "D")

    fc = pd.read_csv(dir / "USPPFlowMonitoring2006_2025.csv")
    daymet = pd.read_csv(dir / "Daymet2006_2025.csv")
    et = pd.read_csv(dir / "Jawad_ET/J.csv")

    rows = []
    dist_rows = []

    sitewise = {}
    for site in sites:
        s = fc[fc["Site"] == site].copy()
        ts = pd.Series(s["Flow Code"].values, index = pd.to_datetime(s["Date"]))
        sitewise[site] = ts
        counts = s["Flow Code"].value_counts(normalize = True)
        dist_rows.append({
            "Site": site,
            "Dry": float(counts.get(0, 0)),
            "Wet": float(counts.get(1, 0)),
            "Flowing": float(counts.get(2, 0)),
            "Dominant": int(counts.idxmax()) if not counts.empty else None,
        })

    for site in sites:
        cam = site_to_camera.get(site)
        fcs = sitewise.get(site, pd.Series(dtype = float))

        ps = pd.Series(dtype = float)
        vs = pd.Series(dtype = float)
        ts = pd.Series(dtype = float)
        ets = pd.Series(dtype = float)

        if "camera" in daymet.columns:
            dcam = daymet[daymet["camera"] == cam].copy()
            if "date" in dcam.columns:
                dcam = dcam.set_index(pd.to_datetime(dcam["date"]))
                if "prcp" in dcam.columns:
                    ps = dcam["prcp"]
                if "tvpd" in dcam.columns:
                    vs = dcam["tvpd"]
                if "tmax" in dcam.columns:
                    ts = dcam["tmax"]
        if "Camera" in et.columns:
            ecam = et[et["Camera"] == cam].copy()
            if "date" in ecam.columns:
                ecam = ecam.set_index(pd.to_datetime(ecam["date"]))
                if "ET" in ecam.columns:
                    ets = ecam["ET"]

        features = {
            "Flow Code": fcs,
            "Precipitation": ps,
            "ET": ets,
            "Temperature": ts,
            "VPD": vs,
        }

        for var, series in features.items():
            missing, longest, gap_type, monthly, sd = missing_metrics(series, full)
        
            mbf = missingness(series, fcs, full) if var != "Flow Code" else {}
            rows.append({
                "Site": site,
                "Variable": var,
                "Fraction Missing": missing,
                "Longest Consecutive Gap": longest,
                "Gap Type": gap_type,
                "Start Date": sd,
                "Monthly Missing Fraction": json.dumps(monthly),
                "Missingness by Flow": json.dumps(mbf),
            })

    out = pd.DataFrame(rows)
    out.to_csv(out_dir / "Missing.csv", index = False)

    flow = pd.DataFrame(dist_rows)
    flow.to_csv(out_dir / "FCDistribution.csv", index = False)

In [33]:
dir = Path("..") / "Data"
out_dir = Path("..") / "Data/Monitoring"
main(dir, out_dir)

In [34]:
out = Path("..") / "Data/Monitoring"
if (out_dir / "FCDistribution.csv").exists():
    print("Flow Distribution:")
    print(pd.read_csv(out / "FCDistribution.csv").to_string(index = False))
if (out / "Missing.csv").exists():
    print("Missing Data Summary:")
    print(pd.read_csv(out_dir / "Missing.csv").drop(columns = ["Monthly Missing Fraction", "Missingness by Flow"]).to_string(index = False))

Flow Distribution:
              Site      Dry      Wet  Flowing  Dominant
          St.David 0.402663 0.124052 0.459127         2
        Contention 0.445963 0.035344 0.518693         2
          Fairbank 0.220718 0.018619 0.760664         2
         Boquillas 0.000000 0.012884 0.970334         2
CharlestonMesquite 0.107960 0.011168 0.880695         2
             Moson 0.000000 0.000000 1.000000         2
            Hunter 0.122073 0.064607 0.813320         2
          Hereford 0.003308 0.001272 0.995420         2
Missing Data Summary:
              Site      Variable  Fraction Missing  Longest Consecutive Gap   Gap Type Start Date
          St.David     Flow Code          0.168116                      577 systematic 2010-06-16
          St.David Precipitation          0.000701                        1     random 2008-12-31
          St.David            ET          0.129417                      923 systematic 2023-01-01
          St.David   Temperature          0.000701             

In [35]:
def print_json(title, data_dict):
    print()
    print(f"{title}:")
    if not data_dict:
        print("    (none)")
        return
    width = max(len(str(k)) for k in data_dict.keys())
    for k, v in data_dict.items():
        print(f"    {str(k).rjust(width)} : {v}")

missing = pd.read_csv(out_dir / "Missing.csv")

for i in range(len(missing)):
    row = missing.iloc[i]

    site = row["Site"]
    var = row["Variable"]

    monthly = json.loads(row["Monthly Missing Fraction"])
    mbf = json.loads(row["Missingness by Flow"])

    print()
    print(f" Site: {site}   Variable: {var}")
    print()

    print(f"  Fraction Missing:        {row["Fraction Missing"]}")
    print(f"  Longest Consecutive Gap: {row["Longest Consecutive Gap"]}")

    if "Gap Start Date" in row:
        print(f"  Gap Start Date:          {row["Gap Start Date"]}")

    print(f"  Gap Type:                {row["Gap Type"]}")

    print_json("Monthly Missing Fraction", monthly)
    print_json("Missingness by Flow", mbf)



 Site: St.David   Variable: Flow Code

  Fraction Missing:        0.1681155356141334
  Longest Consecutive Gap: 577
  Gap Type:                systematic

Monthly Missing Fraction:
     1 : 0.1338709677419355
     2 : 0.1168141592920354
     3 : 0.15
     4 : 0.17666666666666667
     5 : 0.15
     6 : 0.21666666666666667
     7 : 0.13333333333333333
     8 : 0.22241086587436332
     9 : 0.2631578947368421
    10 : 0.14261460101867574
    11 : 0.15789473684210525
    12 : 0.15789473684210525

Missingness by Flow:
    (none)

 Site: St.David   Variable: Precipitation

  Fraction Missing:        0.000701065619742
  Longest Consecutive Gap: 1
  Gap Type:                random

Monthly Missing Fraction:
     1 : 0.0
     2 : 0.0
     3 : 0.0
     4 : 0.0
     5 : 0.0
     6 : 0.0
     7 : 0.0
     8 : 0.0
     9 : 0.0
    10 : 0.0
    11 : 0.0
    12 : 0.008488964346349746

Missingness by Flow:
    -1 : 0.0
     0 : 0.0012557555462536626
     1 : 0.0
     2 : 0.0007342143906020558

 Site: 